# Less is More? A Gentle Critique of Prompt-Based Compression

[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/code/addarm/less-is-More-a-critique-of-prompt-based-compression)


![Grock LLM Landscape](https://raw.githubusercontent.com/adamd1985/quant_research/refs/heads/main/images/caveman_critique_banner.jpg)

<!-- @import "[TOC]" {cmd="toc" depthFrom=1 depthTo=6 orderedList=false} -->

## Introduction

If you're a user of GenAI, you'd experience how loquacious some AI can be, and if you work with Copilot, Claude Code, or any of the newer agent-mode tools, you'd be irked when you ask for a one-line fix and get three paragraphs of preamble, a code block, and a polite summary of what the code block already says! That verbosity isn't free unfrotunately. It costs tokens.

Recently we have seen a rise of solutions aimed to reduce this token waste through verbosity, a knwom one is **Caveman**, which claims to fix chatty agents with a system prompt. Caveman instructs the model to drop articles, filler words, and pleasantries claiming 75% fewer output tokens, full technical accuracy, and a 3x speed improvement. This tool and its claims are worth a deep dive in this companion artcile to our recent LLM deep dive: ["The Road to Agency: How Large Language Models Work"](https://medium.com/call-for-atlas/the-road-to-agency-how-large-language-models-work-94d9907b03af).

Within this light article, we'll look at the scientific basis for why brevity instructions might work. Then, we'll run a few small experiments to see what prompt-based compression actually does to the model's output distribution, perplexity, and task accuracy. Finally, we'll separate what the evidence supports from what it doesn't.

## Notebook Setup

We need a model that actually follows system prompts as Caveman and its like, target Claude Sonnet, GPT-4 and other frontier models that went through heavy RLHF and developed the verbosity bias that Caveman claims to fix. We'll have to use **Qwen2.5-1.5B-Instruct** throughout. It's Alibaba's compact instruct model, trained with RLHF and DPO, genuinely verbose by default, and small enough to run on CPU in bfloat16 (~3 GB, hopefully it runs on your machine). It won't match Claude's capability - but it's close enough proxy.

In [ ]:
# Optional environment bootstrap for a fresh Kaggle or Colab session.
# If you already created the environment from requirements.txt or environment.yml, skip this cell.
# !pip install -q torch==2.11.0 --index-url https://download.pytorch.org/whl/cpu
# !pip install -q transformers==5.6.2 numpy pandas matplotlib tqdm python-dotenv

In [ ]:
import os
import platform

os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "true"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "0"
# os.environ["HF_TOKEN"] = "XXX" # PUT IN YOURS HERE OR USE .ENV


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import transformers as _tf
from dotenv import load_dotenv
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.auto import tqdm

load_dotenv()

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

np.random.seed(42)
torch.manual_seed(42)

print(f"Python {platform.python_version()}  |  PyTorch {torch.__version__}  |  Transformers {_tf.__version__}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

In [ ]:
import contextlib
import io
import logging
import warnings

warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)


MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
HF_TOKEN = os.environ.get("HF_TOKEN")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    token=HF_TOKEN,
)
model.eval()

print(f"Model: {MODEL_ID}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Vocab size: {tokenizer.vocab_size:,}")

## What is Prompt-Based Compression

Tools that allow such compression ship a system prompt that tells the model to write in compressed, article-free English. Drop "the", "a", "an". Skip pleasantries. Use sentence fragments. Keep code blocks intact. The prompt also defines intensity levels (lite through ultra) and includes a "wenyan" variant that instructs the model to respond in classical Chinese.

The entire mechanism is prompt engineering. 

There's no model modification, no post-processing, no custom tokenizer. They operate on the thesis that if you tell the model to be terse, it produces fewer tokens. The question is whether those fewer tokens carry the same information.

Let's set up three system prompts so we can compare them throughout the notebook. The first is a standard helpful-assistant baseline. The second is a single line asking for concise answers. The third is a representative caveman-style prompt that captures the core rules without copying the full 88-line skill file. Follow the code below:

In [ ]:
SYSTEM_BASELINE = "You are a helpful assistant."

SYSTEM_CONCISE = "Answer concisely. No filler, no preamble."

SYSTEM_CAVEMAN = (
    "Reply in compressed caveman speak. Rules:\n"
    "- Drop articles (a, an, the)\n"
    "- Use sentence fragments, not full sentences\n"
    "- No filler, no pleasantries, no hedging\n"
    "- Keep code blocks unchanged\n\n"
    "Example:\n"
    "User: What does a Python decorator do?\n"
    "Assistant: Wraps function. Adds behavior before/after call without modifying original. "
    "Common uses: logging, auth checks, caching. Syntax: @decorator above def.\n\n"
    "User: How does git rebase work?\n"
    "Assistant: Replays commits onto different base. Rewrites history. "
    "Interactive mode (-i) lets you squash/reorder. Never rebase shared branches."
)

SYSTEM_WENYAN = (
    "Respond in classical Chinese (wenyan). Be maximally compressed. "
    "Use literary Chinese, not modern Mandarin. "
    "Keep code blocks and variable names unchanged.\n\n"
    "Example:\n"
    "User: What does a Python decorator do?\n"
    "Assistant: 裝飾器者，包函數而增行為於前後，不改原體。常用於記錄、鑑權、緩存。"
)

PROMPTS = {
    "baseline": SYSTEM_BASELINE,
    "concise": SYSTEM_CONCISE,
    "caveman": SYSTEM_CAVEMAN,
    "wenyan": SYSTEM_WENYAN,
}


We need a generation function and a few helpers for computing log-probabilities. These follow the same pattern from the earlier article, extended to accept a system prompt via the chat template.

In [ ]:
def build_prompt(system, user_msg):
    """Format a system + user message through the chat template."""
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_msg},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def generate(system, user_msg, max_new_tokens=500, temperature=0.7, seed=42):
    """Generate a response given system + user message. Uses KV cache for speed."""
    torch.manual_seed(seed)
    prompt_text = build_prompt(system, user_msg)
    input_ids = tokenizer(prompt_text, return_tensors="pt").input_ids
    prompt_len = input_ids.shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.95,
            use_cache=True,
        )

    response_ids = output_ids[0, prompt_len:]
    response_text = tokenizer.decode(response_ids, skip_special_tokens=True)
    return response_text, len(response_ids)


def sequence_logprob(text):
    """Compute per-token log-probabilities for a string."""
    ids = tokenizer.encode(text, return_tensors="pt")
    with torch.no_grad():
        logits = model(ids, return_dict=True).logits.float()
    logprobs = torch.log_softmax(logits[:, :-1, :], dim=-1)
    targets = ids[:, 1:]
    token_lp = logprobs.gather(dim=-1, index=targets.unsqueeze(-1)).squeeze(-1)
    return token_lp.squeeze(0)


def perplexity(text):
    """Compute perplexity of a string under the model."""
    lp = sequence_logprob(text)
    return torch.exp(-lp.mean()).item()


def token_entropies(text):
    """Compute next-token entropy at each position."""
    ids = tokenizer.encode(text, return_tensors="pt")
    with torch.no_grad():
        logits = model(ids, return_dict=True).logits.float()
    probs = torch.softmax(logits[:, :-1, :], dim=-1)
    ent = -(probs * torch.log2(probs + 1e-12)).sum(dim=-1)
    return ent.squeeze(0)

Now let's see what the three prompting styles actually produce. We'll ask one concrete coding question and put the outputs side by side.

In [ ]:
demo_question = "Why does a Python list comprehension sometimes run faster than a for-loop?"

for label, sys_prompt in tqdm([("baseline", SYSTEM_BASELINE), ("concise", SYSTEM_CONCISE), ("caveman", SYSTEM_CAVEMAN)]):
    text, n_tokens = generate(sys_prompt, demo_question, max_new_tokens=800)
    print(f"--- {label} ({n_tokens} tokens) ---")
    print(text)
    print()

That side-by-side is worth pausing on. The baseline tends to be the longest, the concise prompt shorter, and the caveman style shorter still. But the interesting question isn't whether we can make the model produce fewer tokens. Of course we can. The interesting question is what gets lost when we do.

To answer that, we need to separate several claims that Caveman makes simultaneously. The compression claim says fewer output tokens. The speed claim follows from compression: fewer tokens means less autoregressive generation time. The accuracy claim says nothing important is lost. And the distribution claim says this works across 30+ AI coding agents.

These are different claims, and they need different evidence. Token counts address compression. Task outcomes address accuracy. Perplexity and entropy tell us something about how the model's internal distribution shifts. Let's take them in order.

## Why Brevity Might Help

There's a real scientific story behind the idea that shorter outputs might be just as good as long ones. It starts with how language models are trained.

Modern instruct-tuned models go through reinforcement learning from human feedback (RLHF), where human raters compare candidate outputs and pick the better one. The problem is that raters have a bias: they tend to prefer longer responses, even when the shorter response contains the same information. Singhal et al. (2024) showed that a purely length-based reward signal reproduces most of the downstream RLHF improvements, which is a striking finding. Shen et al. (2023) called this "loose lips sink ships" and proposed decoupling reward from length. Saito et al. (2023) found that GPT-4 itself, when used as an evaluator, prefers longer answers more than human raters do.

The implication is that much of the verbosity we see in LLM outputs is a training artifact. The model learned to sound helpful by being long, not by being informative. If that's true, then stripping out the padding shouldn't lose much substance. Caveman's core thesis is built on this observation, and on that point the literature is supportive.

But the story has limits. Hakim (2026) found that brevity constraints improved accuracy by 26 percentage points on a specific subset of benchmark problems, those where larger models paradoxically underperformed smaller ones due to overelaboration. That's real, but it applies to 7.7% of the test problems, not universally. On reading comprehension tasks (BoolQ), brevity constraints actually hurt performance. And the "brief" condition in Hakim's experiments meant hard word-count caps like "under 50 words," which is more aggressive than caveman's fragment style.

Chain-of-thought reasoning creates another tension. Wei et al. (2022) showed that intermediate reasoning steps dramatically improve accuracy on complex tasks. Caveman compresses the prose around code blocks rather than the code itself, so the code survives. But for tasks where the reasoning steps are expressed in natural language, compression could genuinely hurt. The relationship between output length and output quality depends on what kind of work the model is doing.

## What We Can Actually Measure

Before we run experiments, it helps to be explicit about what each metric does and doesn't tell us.

**Perplexity** measures how well the model predicts a sequence of tokens. Given a token sequence $x_1, x_2, \dots, x_T$, perplexity is:

$$
\text{PPL} = \exp\!\left(-\frac{1}{T}\sum_{t=1}^{T}\log p_\theta(x_t \mid x_{<t})\right)
$$

Lower perplexity means the model finds the text more predictable. If caveman-style text has higher perplexity than standard English, that tells us it's more "surprising" to the model. But Holtzman et al. (2020) showed that maximizing likelihood during decoding leads to bland, repetitive text, not high-quality text. So perplexity is a diagnostic, not a verdict.

**Next-token entropy** measures how spread out the model's probability distribution is at each step:

$$
H_t = -\sum_{v \in V} p_\theta(v \mid x_{<t}) \log_2 p_\theta(v \mid x_{<t})
$$

High entropy means the model is uncertain about what comes next. If a prompting style systematically changes the entropy profile, that's worth knowing, but again it doesn't directly tell us whether the output is correct.

**Token count** is the most direct measure of compression. It's what Caveman's benchmarks actually report.

**Task accuracy** is the only metric that directly addresses whether the output is useful. For that we need questions with checkable answers.

The important teaching point is that these metrics can disagree. An answer can be short, probable, and wrong. Or long, surprising, and correct. We'll see all four metrics together.

Let's make this concrete. Below we take two prompts that say roughly the same thing, one in natural English, the other caveman-compressed. For each token in the sequence, we'll show the model's top-5 predictions, the actual token's log-probability, and the local entropy. This is what PPL and $H$ look like up close.

In [ ]:
def diagnose_prompt(text, label, top_k=5):
    """Show per-token log-prob, top-k alternatives, and entropy for a text."""
    ids = tokenizer.encode(text, return_tensors="pt")
    tokens = [tokenizer.decode([t]) for t in ids[0]]

    with torch.no_grad():
        logits = model(ids, return_dict=True).logits.float()

    # Shift: logits[t] predicts token[t+1]
    logprobs = torch.log_softmax(logits[0, :-1, :], dim=-1)
    probs = torch.softmax(logits[0, :-1, :], dim=-1)
    entropy = -(probs * torch.log2(probs + 1e-12)).sum(dim=-1)

    print(f"{'=' * 70}")
    print(f"  {label}")
    print(f"  Text: {text[:80]}{'...' if len(text) > 80 else ''}")
    print(f"{'=' * 70}")
    print(f"{'Pos':<4} {'Token':<12} {'logP':<8} {'H (bits)':<10} Top-{top_k} predictions")
    print(f"{'-' * 70}")

    token_lps = []
    for t in range(len(tokens) - 1):
        actual_id = ids[0, t + 1].item()
        lp = logprobs[t, actual_id].item()
        h = entropy[t].item()
        token_lps.append(lp)

        # Top-k at this position
        topk_probs, topk_ids = probs[t].topk(top_k)
        topk_strs = [f"{tokenizer.decode([tid.item()]).strip()!r}({tp:.2f})" for tid, tp in zip(topk_ids, topk_probs)]
        topk_display = "  ".join(topk_strs)

        print(f"{t + 1:<4} {tokens[t + 1]!r:<12} {lp:>6.2f}   {h:>6.2f}     {topk_display}")

    # Summary
    mean_lp = np.mean(token_lps)
    ppl_val = np.exp(-mean_lp)
    mean_h = entropy.mean().item()
    print(f"{'-' * 70}")
    print(f"  PPL = exp(-mean logP) = exp({-mean_lp:.3f}) = {ppl_val:.1f}")
    print(f"  Mean entropy = {mean_h:.2f} bits")
    print()
    return ppl_val, mean_h


# Natural English vs caveman-compressed, same semantic content
good_prompt = "Please explain why a Python list comprehension is faster than a for-loop."
bad_prompt = "Explain why list comp faster than for-loop."

ppl_good, h_good = diagnose_prompt(good_prompt, "Natural English (full sentence)")
ppl_bad, h_bad = diagnose_prompt(bad_prompt, "Caveman-compressed (fragments)")

print(f"\n{'=' * 70}")
print(f"  Summary comparison:")
print(f"  {'Natural:':<20} PPL = {ppl_good:>6.1f}   H = {h_good:.2f} bits   tokens = {len(tokenizer.encode(good_prompt))}")
print(f"  {'Caveman:':<20} PPL = {ppl_bad:>6.1f}   H = {h_bad:.2f} bits   tokens = {len(tokenizer.encode(bad_prompt))}")
print(f"{'=' * 70}")

Look at the columns. The `logP` column shows how surprised the model is by each actual token; values closer to 0 mean the model predicted it well. The `H` column shows how uncertain the model was before seeing that token; high entropy means many plausible continuations, low entropy means the model was confident about what comes next.

The top-k predictions reveal the mechanism. At positions where articles or filler words appear in the natural version ("a", "the", "please"), the model assigns high probability to them, because they're predictable padding. The caveman version skips those tokens entirely, jumping straight to content words. The PPL difference tells us how much more "work" the model has to do to predict the compressed version. The entropy difference tells us whether the model's uncertainty changes.

If caveman text has moderately higher PPL but substantially fewer tokens, that's the core trade: you're removing easy-to-predict tokens (low information content) and keeping hard-to-predict ones (high information content). In Shannon's terms, you're stripping redundancy and moving closer to the information-theoretic floor, though we can't claim to reach the entropy rate with a prompt trick alone.

## Does Caveman Text Look Natural to the Model?

Let's start with the diagnostic question: when we hand the model text written in different compression styles, does it find compressed text more surprising? We'll write the same technical content three ways and measure perplexity for each.

In [ ]:
# Matched semantic content in three styles
texts = {
    "normal": (
        "A Python list comprehension is generally faster than a for-loop because "
        "the iteration happens inside the interpreter's optimized C code rather than "
        "through repeated Python bytecode dispatch. The comprehension also avoids "
        "the overhead of calling list.append on every iteration."
    ),
    "caveman": (
        "List comp faster than for-loop. Iteration happens in optimized C code, "
        "not repeated Python bytecode dispatch. Also avoids list.append overhead "
        "each iteration."
    ),
    "wenyan": ("列表推導速於迴圈。迭代行於C底層，非逐次Python字節碼調度。亦免每次list.append之耗。"),
}

rows = []
for style, text in texts.items():
    ppl = perplexity(text)
    n_tok = len(tokenizer.encode(text))
    n_char = len(text)
    rows.append({"style": style, "tokens": n_tok, "chars": n_char, "PPL": round(ppl, 1)})

df_ppl = pd.DataFrame(rows)
print(df_ppl.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))

colors = ["#4878cf", "#e8873a", "#6fba3c"]

axes[0].bar(df_ppl["style"], df_ppl["PPL"], color=colors)
axes[0].set_ylabel("Perplexity")
axes[0].set_title("Model perplexity by writing style")

axes[1].bar(df_ppl["style"], df_ppl["tokens"], color=colors)
axes[1].set_ylabel("Token count")
axes[1].set_title("Tokens for equivalent content")

plt.tight_layout()
plt.show()

The pattern to look for here is whether the compressed styles cost more perplexity than they save in tokens. If caveman text is only slightly more surprising but substantially shorter, the trade looks reasonable from a distributional perspective. If wenyan text is dramatically more surprising and barely shorter in token count (because CJK characters are expensive in an English-trained tokenizer), then the compression claim doesn't hold for that mode.

But remember: even if caveman text has identical perplexity to normal English, that still wouldn't prove the output is equally useful. Perplexity measures fit to the model's training distribution, not task correctness. We need the benchmark for that.

## Entropy Profiles Under Different Prompts

A related question is what happens to the model's uncertainty during generation. If a prompting style consistently produces sharper (lower-entropy) next-token distributions, that means the model is more confident about its outputs. Sharper isn't necessarily better, but it tells us whether the prompt is pushing the model into a different regime.

We'll take the same question from earlier and look at the entropy trace for each prompting condition.

In [ ]:
entropy_question = "Explain what a Python decorator does."

fig, ax = plt.subplots(figsize=(9, 3.5))

for label, sys_prompt, color in [
    ("baseline", SYSTEM_BASELINE, "#4878cf"),
    ("concise", SYSTEM_CONCISE, "#e8873a"),
    ("caveman", SYSTEM_CAVEMAN, "#6fba3c"),
]:
    text, n_tok = generate(sys_prompt, entropy_question, max_new_tokens=200)
    # Compute entropy over the full generated response
    full_text = build_prompt(sys_prompt, entropy_question) + text
    ent = token_entropies(full_text)
    # Take last n_tok positions (the response portion)
    resp_ent = ent[-n_tok:].numpy()
    ax.plot(resp_ent, label=f"{label} (mean={resp_ent.mean():.2f} bits)", color=color, alpha=0.8)

ax.set_xlabel("Token position in response")
ax.set_ylabel("Next-token entropy (bits)")
ax.set_title("Entropy trace by prompting style")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

The entropy trace gives us a rough picture of how the prompting style changes the model's internal confidence distribution. It's a diagnostic that complements perplexity: PPL tells us about fit to existing text, entropy tells us about uncertainty during generation. Neither one tells us about correctness. For that, we need to actually check answers.

## The Benchmark: Does Compression Preserve Utility?

This is the center of the critique. Caveman claims "full technical accuracy" alongside token savings. To test that, we need questions with checkable answers. We'll use a small set of arithmetic, reasoning, and coding questions where the correct answer is unambiguous.

The benchmark is intentionally small. We're running on a 1.5B-parameter model that isn't designed for serious QA, so the results are illustrative rather than definitive. What matters is the methodology: every question gets tested under every prompting condition, so we can compare paired differences rather than point estimates.

A caveat on our scoring: we check whether the expected answer appears as a substring in the model's response. That's a toy-level evaluator. It's unfair to the wenyan condition in particular, because a correct answer in classical Chinese characters won't contain the English substring we're looking for. Keep that asymmetry in mind when reading the accuracy column.

In [ ]:
# Small paired benchmark: question, expected answer substring
benchmark = [
    ("What is 17 * 23?", "391"),
    ("What is 144 / 12?", "12"),
    ("What is the square root of 256?", "16"),
    ("In Python, what does len([1,2,3]) return?", "3"),
    ("What HTTP status code means 'Not Found'?", "404"),
    ("In git, what command stages all changes?", "git add"),
    ("What does the 'self' keyword refer to in a Python class method?", "instance"),
    ("What is the time complexity of binary search?", "log"),
    ("What Python built-in sorts a list in place?", "sort"),
    ("Name the HTTP method used to update a resource.", "PUT"),
]

print(f"Benchmark size: {len(benchmark)} questions")
print(f"Prompting conditions: {list(PROMPTS.keys())}")

In [ ]:
results = []

for question, expected in tqdm(benchmark):
    for label, sys_prompt in PROMPTS.items():
        text, n_tokens = generate(sys_prompt, question, max_new_tokens=200, seed=42)
        correct = expected.lower() in text.lower()
        results.append(
            {
                "question": question[:40],
                "condition": label,
                "tokens": n_tokens,
                "correct": correct,
                "response": text[:120],
            }
        )

df_bench = pd.DataFrame(results)

# Summary table
summary = (
    df_bench.groupby("condition")
    .agg(
        mean_tokens=("tokens", "mean"),
        accuracy=("correct", "mean"),
        n=("correct", "count"),
    )
    .round(2)
)

print(summary.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))

conditions = summary.index.tolist()
colors_map = {"baseline": "#4878cf", "concise": "#e8873a", "caveman": "#6fba3c", "wenyan": "#d65f5f"}
bar_colors = [colors_map.get(c, "#999999") for c in conditions]

axes[0].bar(conditions, summary["mean_tokens"], color=bar_colors)
axes[0].set_ylabel("Mean output tokens")
axes[0].set_title("Compression")

axes[1].bar(conditions, summary["accuracy"], color=bar_colors)
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1.05)
axes[1].set_title("Task accuracy")

plt.tight_layout()
plt.show()

The paired design matters here. Because every question runs under every condition, we can look at paired differences rather than comparing group averages. That's important with a small sample: the absolute accuracy numbers are noisy, but the relative pattern across conditions for the same question is more stable.

If the compression conditions achieve similar accuracy with fewer tokens, that's a genuine win. If accuracy drops, the compression is lossy, and the "full technical accuracy" claim doesn't hold. If the concise one-liner achieves most of the token savings, then the incremental contribution of the full caveman prompt is smaller than the headline suggests.

## The Honest Delta

This is the methodological heart of the critique. Caveman's README reports token savings by comparing against a verbose "helpful assistant" baseline. That's the marketing comparison. The fair comparison is against a simple "Answer concisely" instruction, because that's the trivial alternative anyone could use without installing anything.

The question isn't "does Caveman beat verbose mode?" Obviously it does, and so does any brevity instruction. The question is "what does Caveman's 88-line rule set add beyond plain terseness?" 

In [ ]:
# Compute the honest delta from the benchmark data
baseline_tokens = df_bench[df_bench["condition"] == "baseline"]["tokens"].values
concise_tokens = df_bench[df_bench["condition"] == "concise"]["tokens"].values
caveman_tokens = df_bench[df_bench["condition"] == "caveman"]["tokens"].values

marketing_delta = 1 - caveman_tokens.mean() / baseline_tokens.mean()
honest_delta = 1 - caveman_tokens.mean() / concise_tokens.mean()

print(f"Marketing delta (caveman vs baseline):  {marketing_delta:.1%} fewer tokens")
print(f"Honest delta (caveman vs concise):      {honest_delta:.1%} fewer tokens")
print()

# Bootstrap a simple confidence interval on the marketing delta
n_boot = 5000
rng = np.random.default_rng(42)
boot_deltas = []
for _ in tqdm(range(n_boot)):
    idx = rng.integers(0, len(baseline_tokens), size=len(baseline_tokens))
    boot_deltas.append(1 - caveman_tokens[idx].mean() / baseline_tokens[idx].mean())

lo, hi = np.percentile(boot_deltas, [2.5, 97.5])
print(f"Marketing delta 95% bootstrap CI: [{lo:.1%}, {hi:.1%}]")

This distinction matters. Caveman's benchmark table reports 22% to 87% savings with a 65% average, all measured against a verbose baseline. But the eval code in the Caveman repo actually includes a three-arm design that tests against a concise control. The code comments even acknowledge that comparing only against the verbose baseline "conflates the skill with generic terseness" and call it "cheating." That's intellectually honest engineering, but the README never reports the honest delta. All marketing numbers use the favorable comparison.

Our own benchmark here is tiny and runs on a 1.5B model, so the specific numbers aren't the point. The methodology is: if you want to evaluate a prompting product, compare it against the simplest alternative that addresses the same problem, not just against the worst case.

## Wenyan and the Tokenizer Trap

Caveman includes a "wenyan" mode that instructs the model to respond in classical Chinese. The implied logic is that classical Chinese is informationally dense: a few characters carry a lot of meaning. Fewer characters should mean fewer tokens.

But tokenizers don't count characters. They count subword units, and an English-trained BPE tokenizer was optimized for English byte patterns. CJK characters typically consume 2 to 3 bytes in UTF-8, and each character often becomes its own token or even multiple tokens. Fewer characters can mean more tokens.

In [ ]:
# Compare tokenizer cost for the same concept
pairs = [
    ("New object ref each render", "物出新參照"),
    ("Bug in auth middleware", "認證中間件有誤"),
    ("List comp faster than loop", "列表推導速於迴圈"),
    ("Cache miss causes latency", "快取未中致延遲"),
]

rows = []
for eng, cjk in tqdm(pairs):
    eng_tok = len(tokenizer.encode(eng))
    cjk_tok = len(tokenizer.encode(cjk))
    rows.append(
        {
            "english": eng,
            "eng_chars": len(eng),
            "eng_tokens": eng_tok,
            "wenyan": cjk,
            "cjk_chars": len(cjk),
            "cjk_tokens": cjk_tok,
            "token_ratio": f"{cjk_tok / eng_tok:.1f}x",
        }
    )

df_tok = pd.DataFrame(rows)
print(df_tok[["english", "eng_tokens", "wenyan", "cjk_tokens", "token_ratio"]].to_string(index=False))

If CJK tokens cost more than English tokens for the same semantic content, then wenyan mode doesn't save tokens at all. It saves characters, which might look impressive on screen but doesn't translate to cost or speed savings, because the model processes tokens, not characters. This is a testable claim, and the Caveman repo includes no benchmark measuring wenyan token counts.

## What the Code Tells Us

The Caveman repository contains substantially more engineering than the 88-line skill file. The distribution system is the most impressive part: a tiered installer that detects which of 30+ AI agents are present (Claude Code, Cursor, Windsurf, Gemini CLI, and more) and injects the caveman rules through each agent's native configuration mechanism. There's a CI workflow that propagates changes from one canonical source file to all agent-specific copies. That's real operational engineering that solves a genuinely hard problem.

The security hardening is also substantive. The hook system uses atomic writes with `O_NOFOLLOW`, uid ownership verification, and a temp-plus-rename pattern to prevent symlink attacks at a predictable user-writable path (`~/.claude/.caveman-active`). On shared systems like university machines or CI runners, that path could be targeted by a malicious user planting a symlink. The defense is layered and correct.

The benchmark and eval systems are well-structured too. The benchmark uses real API-reported token counts (ground truth from Claude's tokenizer), and the eval system includes a proper three-arm design. The code comments acknowledge the distinction between the marketing delta and the honest delta. That's a level of methodological self-awareness that most open-source projects don't bother with.

Where the engineering falls short is sample size. Both the benchmark and the eval run on 10 prompts. With n=10 and an observed compression range of 22% to 87%, the standard error on the mean is large. A 95% confidence interval on the average compression ratio would span roughly plus or minus 15 percentage points. The headline "75% savings" could honestly be anywhere from 50% to 80%. The `benchmarks/results/` directory contains only a `.gitkeep` file, so either the results were gitignored or the benchmark hasn't been committed.

## The Cited Paper: Hakim 2026

Caveman's README cites Hakim (2026), *Brevity Constraints Reverse Performance Hierarchies in Language Models*, as scientific support. The citation is accurate but selective.

Hakim evaluated 31 models (0.5B to 405B parameters) across 1,485 problems from five benchmarks: GSM8K, BoolQ, ARC-Easy, CommonsenseQA, and MMLU-STEM. The core finding is that on 7.7% of benchmark problems, larger models underperform smaller ones by 28.4 percentage points. The mechanism is "spontaneous scale-dependent verbosity": large models overelaborate and introduce errors. Constraining them to brief responses improved accuracy by 26 percentage points on those problems and reduced the performance gap by two thirds.

The methodology is solid. Greedy decoding (temperature=0) makes results deterministic and reproducible. The three-condition causal design (control, brief, direct) isolates the effect. Cohen's d = 1.34 is a very large effect size. Three independent contamination tests and Bonferroni correction add further rigor.

But the Caveman README uses this paper to support a broader claim than the paper actually makes. Three things to note:

First, the 26-point improvement applies to 7.7% of problems. On the other 92.3%, the paper doesn't demonstrate a brevity benefit. The improvement is real but narrow.

Second, Hakim's "brief" condition used hard constraints like "under 50 words" for math and "10 words or less" for reading comprehension. Caveman's fragment style still allows multi-paragraph responses. The interventions aren't the same.

Third, the paper tested factual QA and math, not code generation. Code has different information density properties, and Caveman's primary use case is coding agents. The paper supports a weaker but still useful claim: brevity rarely hurts accuracy and sometimes dramatically helps, especially when the model would otherwise overelaborate.

# Conclusion

Caveman's core thesis survives, but in a weaker form than the README suggests. Prompt-based output compression is real, often useful, and probably mostly harmless for coding tasks where the code blocks themselves stay intact. The RLHF verbosity-bias literature gives it genuine scientific grounding: much of the padding in LLM outputs is a training artifact, and stripping it out shouldn't lose much substance.

The stronger claims don't survive as well. The "75% savings" headline is better described as 65% on average, with a range of 22% to 87% and no confidence intervals. The "full technical accuracy" claim is asserted but unmeasured. There's no accuracy evaluation in the repo, only token counts. And the fair comparison isn't against a verbose baseline but against a simple "Answer concisely" instruction, which likely captures most of the token savings without an 88-line rule set.

Our perplexity and entropy experiments provide a diagnostic window into what's happening. Compressed text may be somewhat off-distribution for the model, and different prompting styles do shift the model's uncertainty profile. But those metrics don't settle the accuracy question. The small benchmark gives a paired view of compression versus correctness, and the key observation is that token savings and accuracy preservation are separate claims that need separate evidence.

If there's one takeaway for practitioners, it's this: a one-line instruction like "Answer concisely" gets you most of the way to output compression. Caveman's real contribution may be its distribution engineering across 30+ agents and its operational hardening, not the compression itself. And if there's one takeaway for researchers, it's that evaluating a prompting product requires comparing against the simplest alternative that addresses the same problem. Marketing deltas are interesting. Honest deltas are informative.

## References

1. Hakim, M.A. (2026). *Brevity Constraints Reverse Performance Hierarchies in Language Models*. [arXiv:2604.00025](https://arxiv.org/abs/2604.00025)
2. Singhal, P., Goyal, T., Xu, J., Durrett, G. (2024). *A Long Way to Go: Investigating Length Correlations in RLHF*. COLM 2024. [arXiv:2310.03716](https://arxiv.org/abs/2310.03716)
3. Shen, W., Li, R., Shao, S., et al. (2023). *Loose Lips Sink Ships: Mitigating Length Bias in Reinforcement Learning from Human Feedback*. EMNLP 2023 Findings. [arXiv:2310.05199](https://arxiv.org/abs/2310.05199)
4. Saito, K., Wachi, A., Wataoka, K., Akimoto, Y. (2023). *Verbosity Bias in Preference Labeling by Large Language Models*. [arXiv:2310.10076](https://arxiv.org/abs/2310.10076)
5. Wei, J., Wang, X., Schuurmans, D., et al. (2022). *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models*. NeurIPS 2022. [arXiv:2201.11903](https://arxiv.org/abs/2201.11903)
6. Holtzman, A., Buys, J., Du, L., Forbes, M., Choi, Y. (2020). *The Curious Case of Neural Text Degeneration*. ICLR 2020. [arXiv:1904.09751](https://arxiv.org/abs/1904.09751)
7. Meister, C., Pimentel, T., Wiher, G., Cotterell, R. (2022). *Locally Typical Sampling*. TACL 2022. [arXiv:2202.00666](https://arxiv.org/abs/2202.00666)
8. Cobbe, K., Kosaraju, V., Bavarian, M., et al. (2021). *Training Verifiers to Solve Math Word Problems* (GSM8K). [arXiv:2110.14168](https://arxiv.org/abs/2110.14168)
9. Shannon, C.E. (1948). *A Mathematical Theory of Communication*. Bell System Technical Journal, 27(3), 379-423.
10. Brussee, J. (2025). *Caveman: Prompt-Based Output Compression for AI Coding Agents*. [GitHub](https://github.com/juliusbrussee/caveman)

GitHub repository: [here](https://github.com/adamd1985/quant_research/blob/main/caveman_critique.ipynb)

## Media

All media used, in the form of code or images, are either solely owned by me, acquired through licensing, or part of the public domain and available under Creative Commons terms.